# Reproduce the LLM results (Table 5)

This notebook rebuilds every LLM score in the paper from `data/llm_per_seed_scores.csv`.
That file holds one row per model, dataset, prompt, test set and seed, with the Macro-F1, toxic precision and toxic recall of that run.
The notebook reports the mean ± standard deviation over the seeds 1, 42 and 999, in %.

The column `n` is the number of conversations scored in each run. It is below 1,000 where a model refused or returned no answer, since those conversations are left out of the scores.

In [8]:
%pip install -q pandas

Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd

scores = pd.read_csv("data/llm_per_seed_scores.csv")
METRICS = ["macro_f1", "toxic_precision", "toxic_recall"]
ORDER = ["Claude Opus 5", "GPT-5.6 Sol", "Gemini 3.7 Flash", "Dolphin 3 (8B)", "Claude Haiku 4.5", "GPT-5.4 mini", "Gemini 3.1 Flash-Lite"]

assert (scores.groupby(["model", "dataset", "prompt", "test_set"])["seed"].nunique() == 3).all()
print(f"{len(scores)} runs loaded")

120 runs loaded


In [10]:
agg = scores.groupby(["model", "dataset", "prompt", "test_set"])[METRICS].agg(["mean", "std"])  # std uses ddof=1
agg = (agg * 100).round(2)

def table(dataset):
    t = agg.xs(dataset, level="dataset").reset_index()
    t["model"] = pd.Categorical(t["model"], ORDER, ordered=True)
    t["prompt"] = t["prompt"].map({"base": "Base", "rule": "RULE"})
    t = t.sort_values(["model", "prompt", "test_set"])
    out = pd.DataFrame({"Model": t["model"], "Prompt": t["prompt"], "Test set": t["test_set"]})
    for m, label in zip(METRICS, ["Macro-F1", "Toxic Precision", "Toxic Recall"]):
        out[label] = [f"{a:.2f} ± {b:.2f}" for a, b in zip(t[(m, "mean")], t[(m, "std")])]
    return out.reset_index(drop=True)

In [11]:
print("PLEROMA")
display(table("pleroma"))

PLEROMA


,Model,Prompt,Test set,Macro-F1,Toxic Precision,Toxic Recall
0,Claude Opus 5,Base,Local,66.20 ± 0.09,77.35 ± 2.10,49.13 ± 1.97
1,Claude Opus 5,Base,Shared Global,66.19 ± 1.09,72.19 ± 4.15,55.13 ± 6.18
2,Claude Opus 5,RULE,Local,62.61 ± 1.37,76.50 ± 1.54,42.07 ± 3.69
3,Claude Opus 5,RULE,Shared Global,65.51 ± 2.09,72.35 ± 3.95,53.27 ± 8.62
4,GPT-5.6 Sol,Base,Local,74.19 ± 1.07,81.68 ± 1.38,63.33 ± 3.89
5,GPT-5.6 Sol,Base,Shared Global,73.62 ± 0.97,76.34 ± 4.16,69.40 ± 5.20
6,GPT-5.6 Sol,RULE,Local,69.51 ± 1.22,82.86 ± 1.46,51.97 ± 3.50
7,GPT-5.6 Sol,RULE,Shared Global,69.62 ± 1.50,75.83 ± 5.22,59.52 ± 5.47
8,Gemini 3.7 Flash,Base,Local,70.37 ± 0.30,84.33 ± 1.77,52.64 ± 1.26
9,Gemini 3.7 Flash,Base,Shared Global,70.29 ± 1.02,78.25 ± 5.73,58.43 ± 6.44


In [12]:
print("MASTODON")
display(table("mastodon"))

MASTODON


,Model,Prompt,Test set,Macro-F1,Toxic Precision,Toxic Recall
0,Claude Opus 5,Base,Local,75.69 ± 1.85,76.30 ± 0.69,74.53 ± 4.22
1,Claude Opus 5,Base,Shared Global,74.16 ± 4.98,75.96 ± 2.05,70.73 ± 10.90
2,Claude Opus 5,RULE,Local,73.50 ± 2.17,77.54 ± 2.33,66.53 ± 2.80
3,Claude Opus 5,RULE,Shared Global,71.38 ± 4.90,76.60 ± 1.76,62.33 ± 10.33
4,GPT-5.6 Sol,Base,Local,79.64 ± 2.21,77.97 ± 2.04,82.73 ± 3.58
5,GPT-5.6 Sol,Base,Shared Global,80.64 ± 6.24,78.45 ± 5.03,84.40 ± 7.90
6,GPT-5.6 Sol,RULE,Local,78.55 ± 2.17,80.82 ± 1.22,74.92 ± 3.82
7,GPT-5.6 Sol,RULE,Shared Global,77.77 ± 5.25,80.69 ± 3.92,73.04 ± 8.65
8,Gemini 3.7 Flash,Base,Local,73.87 ± 0.77,84.92 ± 1.76,59.53 ± 1.60
9,Gemini 3.7 Flash,Base,Shared Global,72.41 ± 4.02,83.88 ± 6.61,57.33 ± 3.75
